# TESTING SCRAPY-PLAYWRIGHT DESIGN & IDEAS

## 1. Link Scoring

### A. Test First Method
---

In [3]:
import re

In [4]:
# Clean Text:
def clean_text(text):
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()

In [5]:
# Score Links Candidates by passing in the discovered link
def score_candidate_link(link):
    # Normalize Text: 
    # 1. Break into chunks: text, href, title, and aria
    # 2. Remove big spaces
    # 3. Remove lowercase
    text = clean_text(link.get("text", "")).lower() 
    href = link.get("href", "").lower() 
    title = clean_text(link.get("title", "")).lower()
    aria = clean_text(link.get("aria_label", "")).lower()

    searchable = " ".join((text, href, title, aria)) # union

    # Score Link:
    score = 0
    reasons = []

    weighted_terms = {
        "happy hour": 100,
        "half off": 100,
        "specials": 75,
        "promotions": 70,
        "offers": 70,
        "deals": 65,
        "menu": 50,
        "brunch": 40,
    }

    for term, weight in weighted_terms.items():
        if term in searchable:
            score += weight
            reasons.append(term)

    negative_terms = (
        "gift-card",
        "gift card",
        "rewards",
        "loyalty",
        "sign-in",
        "login",
        "account",
        "careers",
        "privacy",
        "accessibility",
        "nutrition",
        "contact",    
    )

    for term in negative_terms:
        if term in searchable:
            score -= 10
            reasons.append(f"excluded:{term}")

    return score, reasons

In [6]:
# Link Examples
# Yardhouse Happy Hour
link_1 = "https://www.yardhouse.com/menu-listing/happy-hour"
# Lazy Dog Menu
link_2 = "https://orders.lazydogrestaurants.com/menu"

score_candidate_link(link_1)

AttributeError: 'str' object has no attribute 'get'

## Regex
`re.compile()` converts a regular expression pattern string into a reusable regex pattern object.

In [1]:
import re

In [5]:
nav = "happy hour menu, HAPPY HOUR, HAppy Hour, fffdkf fdhapffdapy fhour."
text = "Alice's favorite numbers are 456, 789, and 1233."
# 1. Compile the regex pattern (matching 3 consecutive digits)
digit_pattern = re.compile(r'\d{3}')

happy_hour = re.compile("happy hour", re.I)

# 2. Use compiled object's findall() method directly
results_1 = digit_pattern.findall(text)
results_2 = happy_hour.findall(nav)
print(results_2)

['happy hour', 'HAPPY HOUR', 'HAppy Hour']


## Load Records

In [5]:
import re
import csv
import random
import pandas as pd

In [3]:
csv_path = "data/src/grok_golden_model.csv"
limit = 10

In [9]:
required = [
    "venue_name",
    "Source URL",
    "Business Type",
    "Incentive Category",
    "Incentive Teaser",
    "Full Incentive Description",
]

candidates = []
seen_urls = set()

with open(csv_path, newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    # self.logger.info("CSV columns: %s", reader.fieldnames)

    total_rows = 0
    missing_required = 0
    bad_url = 0
    duplicate_url = 0

    for row in reader:
        total_rows += 1

        if not all((row.get(col) or "").strip() for col in required):
            missing_required += 1
            continue

        url = row["Source URL"].strip()

        if not url.startswith("http") or re.search(r"\s", url):
            bad_url += 1
            continue

        if url in seen_urls:
            duplicate_url += 1
            continue

        seen_urls.add(url)
        candidates.append(row)

    # rng = random.Random(getattr(self, "random_seed", 42))
    # rng.shuffle(candidates)
    output = candidates[:limit]
    df_output = pd.DataFrame(output)
df_output
    # print(output)
    # print(type(output))
    # print(len(output))

    # self.logger.info("Total CSV rows seen: %d", total_rows)
    # self.logger.info("Rows skipped missing required fields: %d", missing_required)
    # self.logger.info("Rows skipped bad URL: %d", bad_url)
    # self.logger.info("Rows skipped duplicate URL: %d", duplicate_url)
    # self.logger.info("Candidate rows after cleaning: %d", len(candidates))
    # self.logger.info("Rows selected for scraping: %d", len(output))

    # for r in output:
    #     self.logger.info("Selected venue: %s | %s", r.get("venue_name"), r.get("Source URL"))
    # return output


,venue_name,Source URL,Business Type,Incentive Category,Incentive Teaser,Full Incentive Description
0,NOVA Kitchen & Bar,https://www.novaoc.com/,Restaurant,Happy Hour,Roll a 7 â€“ Entire Check FREE,"Sushi, bites, and cocktails from Bar Menu. Rol..."
1,Puesto,https://www.eatpuesto.com/,Mexican Restaurant,Taco Tuesday / Happy Hour,Half-Priced Tacos on Tuesdays,All tacos half price on Taco Tuesday (includin...
2,Bowlero Orange County,https://www.bowlero.com/location/bowlero-orang...,Entertainment (Bowling),Group Rate / Night Strike,Unlimited Bowling Special,Unlimited bowling + shoe rental for $24.99 dur...
3,Regal Edwards (multiple OC locations),https://www.regmovies.com/groups-and-events/gr...,Entertainment,Group Rate,Group Movie Ticket Discount (21+),Discounted tickets for groups of 21 or more: $...
4,Citrus City Grille,https://citruscitygrille.com/,Restaurant,Happy Hour,$10 Menu â€“ All Items,"Artisan-crafted cocktails, premium wines, loca..."
5,Lazy Dog Restaurant & Bar,https://lazydogrestaurants.com/pages/orange-ca,American Restaurant,Happy Hour,Happy Hour Bites & Drinks,"Discounted appetizers, burgers, and drinks dur..."
6,The Lab Anti-Mall,https://www.thelab.com/,Entertainment / Dining Complex,Happy Hour,Multiple Venue Happy Hour Deals,Participating spots (including Ruin Bar and ot...
7,Rancho Capistrano Winery,https://www.ranchocapwinery.com/,Winery / Restaurant,Happy Hour,Wine Happy Hour Flights & Menu,Discounted wine flights and happy-hour menu it...
8,Yard House,https://www.yardhouse.com/locations,Restaurant / Bar,Happy Hour,Half-Price Appetizers & Drinks,Select appetizers and drinks at half price dur...
9,Seasons 52,https://www.seasons52.com/locations/ca/costa-mesa,Restaurant,Happy Hour,Bar Menu Specials,Discounted wines by the glass and small plates...


## JSON Conversion
* Converts `grok_golden_model.json` to `grok_golden_model.csv`
* Uses:
    * `json_extract.py` file functions
    * `output_model_venue(1)` & `json_conv_csv(2)`
* **OUTPUT:** `grok_golden_model.csv`

In [1]:
import pandas as pd
import sys
import os
# Use current working directory 
sys.path.insert(0, os.getcwd())
sys.path.append(os.path.abspath('..'))

In [2]:
from json_extract import (
    output_model_venue,
    json_conv_csv
)

## Working with `scrapy_output.json`

In [3]:
OUTPUT = "scrapy_output.json"

In [5]:
out = output_model_venue(OUTPUT)

[
    {
        "className": "css-175oi2r r-1mdbw0j r-wk8lta",
        "words": 40,
        "text": "HALF OFF\nFour Cheese Spinach Dip\nMiguel's Queso Dip\nPoke Nachos*\nChicken Nachos\nCrispy Brussels Sprouts\nChicken Lettuce Wraps\nHand-Battered Chicken Tenders\nWisconsin Fried Cheese Curds\nClassic Sliders*\nBlackened Ahi Sashimi*\nFried Calamari\nBoneless Wings\nGardein\u00ae Wings\nSpicy Za'atar Hummus",
        "children": [
            "HALF OFF\nFour Cheese Spinach Dip\nMiguel's Queso Dip\nPoke Nachos*\nChicken Nachos\nCrispy Brussels Sprouts\nChicken Lettuce Wraps\nHand-Battered Chicken Tenders\nWisconsin Fried Cheese Curds\nClassic Sliders*\nBlackened Ahi Sashimi*\nFried Calamari\nBoneless Wings\nGardein\u00ae Wings\nSpicy Za'atar Hummus"
        ]
    },
    {
        "className": "css-175oi2r r-1udh08x r-13qz1uu",
        "words": 40,
        "text": "HALF OFF\nFour Cheese Spinach Dip\nMiguel's Queso Dip\nPoke Nachos*\nChicken Nachos\nCrispy Brussels Sprouts\nChicken Lettuc

In [42]:
df_output = json_conv_csv(out, target_col=["className", "words", "text", "children"])

In [61]:
df_output

,className,words,text,children
0,css-175oi2r r-1mdbw0j r-wk8lta,40,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...
1,css-175oi2r r-1udh08x r-13qz1uu,40,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...
2,css-175oi2r,40,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...
3,css-175oi2r r-184en5c,40,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...
4,css-175oi2r r-1awozwy r-1777fci r-1uwte3a r-1h...,40,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...
5,css-175oi2r r-1habvwh r-6koalj r-eqz5dr r-1jj8...,40,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...,HALF OFF
5,css-175oi2r r-1habvwh r-6koalj r-eqz5dr r-1jj8...,40,HALF OFF\nFour Cheese Spinach Dip\nMiguel's Qu...,Four Cheese Spinach Dip\nMiguel's Queso Dip\nP...
6,css-175oi2r r-1mdbw0j r-wk8lta,13,HALF OFF\nThree Cheese\nLoaded Pepperoni\nMarg...,HALF OFF\nThree Cheese\nLoaded Pepperoni\nMarg...
7,css-175oi2r r-1awozwy r-1pz39u2 r-96ba3z r-18u...,13,HALF OFF\nThree Cheese\nLoaded Pepperoni\nMarg...,HALF OFF\nThree Cheese\nLoaded Pepperoni\nMarg...
8,css-175oi2r r-1awozwy r-1pz39u2 r-18u37iz r-1b...,13,HALF OFF\nThree Cheese\nLoaded Pepperoni\nMarg...,HALF OFF


In [ ]:
df_output = df_output.explode("children")
df_children = df_output.drop_duplicates(subset=["children"]) # Not the most helpful
df_text = df_output.drop_duplicates(subset=["text"])     # Might be the most reliable
df_output

child = df_children["children"].tolist()
print(len(child))
print(child)
print(child[2])

5
["HALF OFF\nFour Cheese Spinach Dip\nMiguel's Queso Dip\nPoke Nachos*\nChicken Nachos\nCrispy Brussels Sprouts\nChicken Lettuce Wraps\nHand-Battered Chicken Tenders\nWisconsin Fried Cheese Curds\nClassic Sliders*\nBlackened Ahi Sashimi*\nFried Calamari\nBoneless Wings\nGardein® Wings\nSpicy Za'atar Hummus", 'HALF OFF', "Four Cheese Spinach Dip\nMiguel's Queso Dip\nPoke Nachos*\nChicken Nachos\nCrispy Brussels Sprouts\nChicken Lettuce Wraps\nHand-Battered Chicken Tenders\nWisconsin Fried Cheese Curds\nClassic Sliders*\nBlackened Ahi Sashimi*\nFried Calamari\nBoneless Wings\nGardein® Wings\nSpicy Za'atar Hummus", 'HALF OFF\nThree Cheese\nLoaded Pepperoni\nMargherita\nThe Carnivore\nTruffled Mushroom\nBuffalo Chicken', 'Three Cheese\nLoaded Pepperoni\nMargherita\nThe Carnivore\nTruffled Mushroom\nBuffalo Chicken']
Four Cheese Spinach Dip
Miguel's Queso Dip
Poke Nachos*
Chicken Nachos
Crispy Brussels Sprouts
Chicken Lettuce Wraps
Hand-Battered Chicken Tenders
Wisconsin Fried Cheese Curds

In [7]:
GOLDEN_SOURCE_JSON = "data/src/grok_golden_model.json"

In [ ]:
grok_venues = output_model_venue(GOLDEN_SOURCE_JSON)

NameError: name 'GOLDEN_SOURCE_JSON' is not defined

In [17]:
required = [
    "venue_name",
    "Source URL",
    "Business Type",
    "Incentive Category",
    "Incentive Teaser",
    "Full Incentive Description",   
]

In [12]:
target_grok = [
    "venue_name",
    "source_url",
    "business_type",
    "incentive_category",
    "incentive_teaser",
    "full_incentive_description",
]

In [13]:
df_grok = json_conv_csv(grok_venues, target_col=target_grok)
df_grok

,venue_name,source_url,business_type,incentive_category,incentive_teaser,full_incentive_description
0,NOVA Kitchen & Bar,https://www.novaoc.com/,Restaurant,Happy Hour,Roll a 7 â€“ Entire Check FREE,"Sushi, bites, and cocktails from Bar Menu. Rol..."
1,Puesto,https://www.eatpuesto.com/,Mexican Restaurant,Taco Tuesday / Happy Hour,Half-Priced Tacos on Tuesdays,All tacos half price on Taco Tuesday (includin...
2,Bowlero Orange County,https://www.bowlero.com/location/bowlero-orang...,Entertainment (Bowling),Group Rate / Night Strike,Unlimited Bowling Special,Unlimited bowling + shoe rental for $24.99 dur...
3,Regal Edwards (multiple OC locations),https://www.regmovies.com/groups-and-events/gr...,Entertainment,Group Rate,Group Movie Ticket Discount (21+),Discounted tickets for groups of 21 or more: $...
4,Citrus City Grille,https://citruscitygrille.com/,Restaurant,Happy Hour,$10 Menu â€“ All Items,"Artisan-crafted cocktails, premium wines, loca..."
5,Lazy Dog Restaurant & Bar,https://lazydogrestaurants.com/pages/orange-ca,American Restaurant,Happy Hour,Happy Hour Bites & Drinks,"Discounted appetizers, burgers, and drinks dur..."
6,The Lab Anti-Mall,https://www.thelab.com/,Entertainment / Dining Complex,Happy Hour,Multiple Venue Happy Hour Deals,Participating spots (including Ruin Bar and ot...
7,Rancho Capistrano Winery,https://www.ranchocapwinery.com/,Winery / Restaurant,Happy Hour,Wine Happy Hour Flights & Menu,Discounted wine flights and happy-hour menu it...
8,Yard House,https://www.yardhouse.com/locations,Restaurant / Bar,Happy Hour,Half-Price Appetizers & Drinks,Select appetizers and drinks at half price dur...
9,Seasons 52,https://www.seasons52.com/locations/ca/costa-mesa,Restaurant,Happy Hour,Bar Menu Specials,Discounted wines by the glass and small plates...


In [18]:
df_grok.columns = required

In [20]:
df_grok.to_csv('C:\\Users\\User\\Desktop\\David_Homework\\Scraper\\Venue-Scraper\\scrapy_project\\venue_scraper\\data\\src\\grok_golden_model.csv', index=False)

## Python args & kwargs
Allows a function to accept an arbitrary number of arguments.

In [2]:
def total_sum(*args):
    print(args)
    return sum(args)
print(total_sum(1,2,3,4))

(1, 2, 3, 4)
10


In [4]:
def build_profile(**kwargs):
    print(kwargs)
    for key, value in kwargs.items():
        print(f"{key}: {value}")
build_profile(name="Alice", role="Dev", mow="MOW")

{'name': 'Alice', 'role': 'Dev', 'mow': 'MOW'}
name: Alice
role: Dev
mow: MOW


## Python `super()`

In [1]:
class Parent:
    def show(self):
        print("This is Parent class method")
class Child(Parent):
    def show(self):
        super().show()
        print("This is Child class method")

obj = Child()

In [3]:
obj.show()

This is Parent class method
This is Child class method


In [1]:
class Emp:
    def __init__(self, id, name):
        self.id = id
        self.name = name
class fun(Emp):
    def __init__(self, id, name, email):
        super().__init__(id, name)
        self.email = email
obj = fun(101, "Olivia", "olivia@email.com")
print(obj.id, obj.name, obj.email)

101 Olivia olivia@email.com
